# 3. Распространение и детектирование света

Фотон распространяется по прямой без рассеяния и поглощения внутри сцинтиллятора. При первом достижении границы сферы он поглощается; затем проверяется попадание в один из фотокатодов. Число фотоэлектронов на попавший фотон разыгрывается из распределения Пуассона со средним `a`.

In [1]:
%pip install -U numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from math import cos, sin, acos, pi, sqrt
import numpy as np

def detectors(n=40):
    points = []
    for i in range(n):
        phi = acos(1 - 2 * (i + 0.5) / n)
        theta = pi * (1 + sqrt(5)) * i
        points.append((cos(theta) * sin(phi), sin(theta) * sin(phi), cos(phi)))
    return points

class Detector:
    def __init__(self):
        self.R = 200.0
        self.N = 40
        self.r = 10.0
        self.a = 0.5
        self.pmts = detectors(self.N)

    def get_pmt_positions(self):
        return [(self.R * x, self.R * y, self.R * z) for x, y, z in self.pmts]

    def belongs_to_pmt(self, point):
        x, y, z = point
        if abs(sqrt(x**2 + y**2 + z**2) - self.R) > 1e-4:
            return False
        for center_x, center_y, center_z in self.get_pmt_positions():
            normal_x, normal_y, normal_z = center_x / self.R, center_y / self.R, center_z / self.R
            normal_distance = x * normal_x + y * normal_y + z * normal_z
            if normal_distance <= 0:
                continue
            tangent_distance = sqrt(
                (x - normal_distance * normal_x) ** 2
                + (y - normal_distance * normal_y) ** 2
                + (z - normal_distance * normal_z) ** 2
            )
            if tangent_distance <= self.r:
                return True
        return False

    def generate_photoelectrons(self):
        return int(np.random.poisson(lam=self.a))

    def ray_hits_pmt(self, source_position, direction):
        x, y, z = source_position
        nx, ny, nz = direction
        a = nx**2 + ny**2 + nz**2
        if a == 0:
            raise ValueError("Вектор направления должен быть ненулевым")
        b = 2 * (x * nx + y * ny + z * nz)
        c = x**2 + y**2 + z**2 - self.R**2
        discriminant = b**2 - 4 * a * c
        if discriminant < 0:
            return False
        root = sqrt(discriminant)
        intersections = [t for t in ((-b - root) / (2 * a), (-b + root) / (2 * a)) if t >= 0]
        if not intersections:
            return False
        t = min(intersections)
        hit_point = (x + t * nx, y + t * ny, z + t * nz)
        return self.belongs_to_pmt(hit_point)

In [3]:
class Source:
    def __init__(self, Y=10000):
        self.Y = Y
        self.rng = np.random.default_rng()

    def generate_directions(self, E):
        if E < 0:
            raise ValueError("Энергия должна быть неотрицательной")
        count = round(self.Y * E)
        cos_theta = self.rng.uniform(-1.0, 1.0, count)
        phi = self.rng.uniform(0.0, 2 * np.pi, count)
        sin_theta = np.sqrt(1.0 - cos_theta**2)
        return np.column_stack((sin_theta * np.cos(phi), sin_theta * np.sin(phi), cos_theta))

In [ ]:
detector = Detector()
source = Source()
source.rng = np.random.default_rng(42)
np.random.seed(42)
energy = 1.0
source_position = (0.0, 0.0, 0.0)
directions = source.generate_directions(energy)
photoelectrons = [
    detector.generate_photoelectrons()
    for direction in directions
    if detector.ray_hits_pmt(source_position, direction)
]
print(f"Испущено фотонов: {len(directions)}")
print(f"Собранный заряд: {sum(photoelectrons)} фотоэлектронов")
print(f"Ожидаемый заряд: {detector.a * source.Y * detector.N * detector.r**2 / (4 * detector.R**2):.1f}")

Испущено фотонов: 10000
Собранный заряд: 112 фотоэлектронов
Ожидаемый заряд: 125.0
